In [1]:
from sdk_client import Robot
rob = Robot("eth0")
rob.hanged_boot()

damp         -> FSM 1   mode 0
stand_up     -> FSM 4   mode 1
height 0.02 m -> FSM 4   mode 1
height 0.04 m -> FSM 4   mode 1
height 0.06 m -> FSM 4   mode 1
height 0.08 m -> FSM 4   mode 1
height 0.10 m -> FSM 4   mode 1
height 0.12 m -> FSM 4   mode 0
height 0.14 m -> FSM 4   mode 0
height 0.16 m -> FSM 4   mode 0
height 0.18 m -> FSM 4   mode 0
height 0.20 m -> FSM 4   mode 0
height 0.22 m -> FSM 4   mode 0
balance      -> FSM 4   mode 0
height_ok    -> FSM 4   mode 0
start        -> FSM 4   mode 0
balanced     -> FSM 501   mode 0


In [ ]:
# from types import SimpleNamespace
# from scripts.shake_hand_trajectory_cli import run_replay

# run_replay(SimpleNamespace(
#     file="scripts/saved_shake_hand_trajectories.json", arm="right", phase="raise",
#     index=None, name=None, rate_hz=50.0, max_increment_rad=0.01,
#     kp=30.0, kd=1.5, waist_kp=30.0, waist_kd=1.5,
#     hold_seconds=1.0, hold_forever=False,
# ))

In [ ]:
import time
import numpy as np

from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

RIGHT_ARM_JOINTS = [22, 23, 24, 25, 26, 27, 28]
NOT_USED_IDX = 29
COMMAND_RATE_HZ = 50.0
RAMP_DURATION_S = 4.0

ChannelFactoryInitialize(int(rob.domain_id), str(rob.iface))

lowstate_type = None
for module_path in (
    "unitree_sdk2py.idl.unitree_hg.msg.dds_",
    "unitree_sdk2py.idl.unitree_go.msg.dds_",
):
    try:
        module = __import__(module_path, fromlist=["LowState_"])
    except Exception:
        continue
    if hasattr(module, "LowState_"):
        lowstate_type = getattr(module, "LowState_")
        break
if lowstate_type is None:
    raise RuntimeError("Could not resolve LowState_ type.")
arm_positions = None

def _right_arm_state_cb(msg):
    global arm_positions
    try:
        arm_positions = np.asarray([float(msg.motor_state[j].q) for j in RIGHT_ARM_JOINTS], dtype=np.float32)
    except Exception:
        return

state_sub = ChannelSubscriber("rt/lowstate", lowstate_type)
state_sub.Init(_right_arm_state_cb, 100)

deadline = time.time() + 3.0
while arm_positions is None and time.time() < deadline:
    time.sleep(0.02)
if arm_positions is None:
    raise TimeoutError("Timed out waiting for right arm joint positions from rt/lowstate.")

pub = ChannelPublisher("rt/arm_sdk", LowCmd_)
pub.Init()
crc = CRC()
cmd = unitree_hg_msg_dds__LowCmd_()
cmd.mode_pr = 0
cmd.mode_machine = 0
cmd.motor_cmd[NOT_USED_IDX].q = 1.0
for joint in RIGHT_ARM_JOINTS:
    cmd.motor_cmd[joint].mode = 1

start_pose = arm_positions.copy()
target_pose = start_pose.copy()
# Extend forward by opening the elbow while leaving the waist to balanced stand.
target_pose[3] = np.float32(0.03)
target_pose[1] = np.float32(min(float(target_pose[1]) + 0.25, 0.25))
target_pose[2] = np.float32(0.0)
target_pose[4] = np.float32(0.0)
target_pose[5] = np.float32(0.0)
target_pose[6] = np.float32(0.0)

steps = max(1, int(RAMP_DURATION_S * COMMAND_RATE_HZ))
dt = 1.0 / COMMAND_RATE_HZ

print("Current right arm pose:", [round(float(v), 3) for v in start_pose.tolist()])
print("Target right arm pose:", [round(float(v), 3) for v in target_pose.tolist()])
print("Only right arm joints are commanded on rt/arm_sdk; the waist stays under balanced-stand control.")

for step_idx in range(1, steps + 1):
    alpha = float(step_idx) / float(steps)
    pose = start_pose + (target_pose - start_pose) * alpha
    for joint, q_val in zip(RIGHT_ARM_JOINTS, pose):
        mc = cmd.motor_cmd[joint]
        mc.mode = 1
        mc.q = float(q_val)
        mc.dq = 0.0
        mc.kp = 30.0
        mc.kd = 1.5
        mc.tau = 0.0
    cmd.crc = crc.Crc(cmd)
    pub.Write(cmd)
    time.sleep(dt)

print("Right arm moved to the forward pose. Re-run the cell if you want to send it again.")


In [ ]:
import time
import numpy as np

from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

RIGHT_ARM_JOINTS = [22, 23, 24, 25, 26, 27, 28]
WAIST_JOINTS = [12, 13, 14]
STATE_JOINTS = RIGHT_ARM_JOINTS + WAIST_JOINTS
NOT_USED_IDX = 29
COMMAND_RATE_HZ = 50.0
WAIST_LOCK_HOLD_S = 1.0
CLEARANCE_RAMP_S = 1.5
RAISE_RAMP_S = 2.5
FINAL_HOLD_S = 2.0

ChannelFactoryInitialize(int(rob.domain_id), str(rob.iface))

lowstate_type = None
for module_path in (
    "unitree_sdk2py.idl.unitree_hg.msg.dds_",
    "unitree_sdk2py.idl.unitree_go.msg.dds_",
):
    try:
        module = __import__(module_path, fromlist=["LowState_"])
    except Exception:
        continue
    if hasattr(module, "LowState_"):
        lowstate_type = getattr(module, "LowState_")
        break
if lowstate_type is None:
    raise RuntimeError("Could not resolve LowState_ type.")

joint_positions = None

def _state_cb(msg):
    global joint_positions
    try:
        joint_positions = np.asarray([float(msg.motor_state[j].q) for j in STATE_JOINTS], dtype=np.float32)
    except Exception:
        return

state_sub = ChannelSubscriber("rt/lowstate", lowstate_type)
state_sub.Init(_state_cb, 100)

deadline = time.time() + 3.0
while joint_positions is None and time.time() < deadline:
    time.sleep(0.02)
if joint_positions is None:
    raise TimeoutError("Timed out waiting for right arm and waist joint positions from rt/lowstate.")

pub = ChannelPublisher("rt/arm_sdk", LowCmd_)
pub.Init()
crc = CRC()
cmd = unitree_hg_msg_dds__LowCmd_()
cmd.mode_pr = 0
cmd.mode_machine = 0
cmd.motor_cmd[NOT_USED_IDX].q = 1.0
for joint in STATE_JOINTS:
    cmd.motor_cmd[joint].mode = 1

current_arm = joint_positions[: len(RIGHT_ARM_JOINTS)].copy()
locked_waist = joint_positions[len(RIGHT_ARM_JOINTS):].copy()

clearance_arm = current_arm.copy()
# Step 2: move the shoulder slightly outboard first so the hand clears the thigh.
clearance_arm[1] = np.float32(max(float(clearance_arm[1]) - 0.35, -0.60))

# Step 3: then move into the raised-hand pose.
raised_arm = np.asarray([-0.103, -0.066, 0.105, 0.187, 0.159, 0.787, -0.065], dtype=np.float32)

def _publish_pose(arm_pose, waist_pose):
    for joint, q_val in zip(RIGHT_ARM_JOINTS, arm_pose):
        mc = cmd.motor_cmd[joint]
        mc.mode = 1
        mc.q = float(q_val)
        mc.dq = 0.0
        mc.kp = 30.0
        mc.kd = 1.5
        mc.tau = 0.0
    for joint, q_val in zip(WAIST_JOINTS, waist_pose):
        mc = cmd.motor_cmd[joint]
        mc.mode = 1
        mc.q = float(q_val)
        mc.dq = 0.0
        mc.kp = 30.0
        mc.kd = 1.5
        mc.tau = 0.0
    cmd.crc = crc.Crc(cmd)
    pub.Write(cmd)

def _hold_pose(arm_pose, waist_pose, seconds):
    steps = max(1, int(max(0.1, float(seconds)) * COMMAND_RATE_HZ))
    dt = 1.0 / COMMAND_RATE_HZ
    for _ in range(steps):
        _publish_pose(arm_pose, waist_pose)
        time.sleep(dt)

def _ramp_pose(start_arm, stop_arm, waist_pose, seconds):
    steps = max(1, int(max(0.1, float(seconds)) * COMMAND_RATE_HZ))
    dt = 1.0 / COMMAND_RATE_HZ
    for step_idx in range(1, steps + 1):
        alpha = float(step_idx) / float(steps)
        arm_pose = start_arm + (stop_arm - start_arm) * alpha
        _publish_pose(arm_pose, waist_pose)
        time.sleep(dt)

print("Current right arm pose:", [round(float(v), 3) for v in current_arm.tolist()])
print("Locked waist pose:", [round(float(v), 3) for v in locked_waist.tolist()])
print("Clearance pose:", [round(float(v), 3) for v in clearance_arm.tolist()])
print("Raised pose:", [round(float(v), 3) for v in raised_arm.tolist()])
print("Stage 1: lock waist. Stage 2: move shoulder away from body. Stage 3: move to raised-hand pose.")

_hold_pose(current_arm, locked_waist, WAIST_LOCK_HOLD_S)
_ramp_pose(current_arm, clearance_arm, locked_waist, CLEARANCE_RAMP_S)
_ramp_pose(clearance_arm, raised_arm, locked_waist, RAISE_RAMP_S)
_hold_pose(raised_arm, locked_waist, FINAL_HOLD_S)

print("Safe staged right-hand raise complete.")
